# ATRA-4B — QLoRA fine-tune on Kaggle (T4 x2)

**Status of this notebook:** written by the ATRA maintainers, **not yet executed on Kaggle**. Run it top to bottom with *Save & Run All*; if a cell fails, the error is real and the fix belongs in the repository, not in a claim that it worked.

What it does, in order:

1. checks the GPU and installs the pinned training dependencies on top of Kaggle's PyTorch;
2. clones the ATRA repository and builds the synthetic dataset deterministically (seed 42);
3. runs the dataset quality gates (leakage, duplicates, chronological split, refusal ratio);
4. fine-tunes `unsloth/Qwen3-4B-Instruct-2507-bnb-4bit` with QLoRA (`train.py`, config `config/default.yaml`);
5. merges the adapter into `Qwen/Qwen3-4B-Instruct-2507` and converts to GGUF for Ollama;
6. leaves everything under `/kaggle/working/atra-export` so you can download it.

Settings: **Accelerator = GPU T4 x2**, **Internet = On**, **Persistence = Files only** (optional). A phone-verified Kaggle account is required for GPU sessions. Free quota is 30 GPU-hours/week.

Nothing in this notebook needs an API key. The base model is Apache-2.0 and downloads anonymously from Hugging Face. Set `ATRA_PUSH`/`HF_TOKEN` only if you want to upload the adapter to your own private Hugging Face repo (off by default).

In [ ]:
!nvidia-smi
import torch, platform
print("torch", torch.__version__, "cuda", torch.version.cuda, "bf16", torch.cuda.is_bf16_supported())
print(platform.python_version())

## 1. Dependencies

Kaggle ships a CUDA build of PyTorch (2.10.0+cu128 on 2026-09-20); keep it. `train.py` uses plain `transformers` + `peft` + `trl` + `bitsandbytes`, so `unsloth` (which pins its own torch) is **not** installed here.

Versions verified against PyPI on 2026-09-20. The first run of this notebook failed here because `transformers==4.62.1` and `huggingface_hub==0.38.2` had never been published; 4.57.6 is the highest 4.x, which is what `trl==0.27.0` expects (`>=4.56.2`).

In [ ]:
%%bash
set -e
pip install -q "transformers==4.57.6" "trl==0.27.0" "peft==0.19.1" "accelerate==1.12.0"   "bitsandbytes==0.50.2" "datasets==4.5.0" "sentencepiece==0.2.1" "huggingface_hub==0.36.2" "pyyaml==6.0.3"
python -c "import transformers, trl, peft, bitsandbytes; print('deps ok', transformers.__version__, trl.__version__, peft.__version__, bitsandbytes.__version__)"

## 2. Get the code

In [ ]:
%%bash
set -e
mkdir -p /kaggle/temp
cd /kaggle/temp
rm -rf ATRA
git clone --depth 1 https://github.com/ARTA-4B/ATRA.git
cd ATRA/model/atra-4b
git -C /kaggle/temp/ATRA rev-parse HEAD
ls

## 3. Build and check the dataset (deterministic)

`--per-domain 200` gives roughly 1,000 examples. The checks refuse to continue if fewer than 40% of examples are `NO_ACTION`, if any example leaks its outcome into the prompt, or if train/validation/test overlap.

In [ ]:
%%bash
set -e
cd /kaggle/temp/ATRA/model/atra-4b
python -m data.build --seed 42 --per-domain 200 --out data/out
python -m data.checks data/out

## 4. Dry run (no model loaded)

In [ ]:
%%bash
set -e
cd /kaggle/temp/ATRA/model/atra-4b
python train.py --dry-run

## 5. Train

T4 has no bf16, so this runs in fp16. `train.py` decides that from the GPU's compute capability, not from `torch.cuda.is_bf16_supported()`, which answers True on a T4 and would select emulated bf16. `ATRA_SEQ` caps the sequence length; 2048 is the config default and should fit a T4 (16 GB) at batch 1 with gradient checkpointing. If you hit CUDA OOM, set `ATRA_SEQ=1024` and rerun. `ATRA_AMP` chooses the precision: `off` (float32, the default here after five runs of dtype disagreements), `fp16`, or `bf16` on Ampere and newer. Expect roughly 1–3 hours for 2 epochs on ~1,000 examples; the notebook's 12-hour limit is far away.

In [ ]:
%%bash
set -e
set -o pipefail
cd /kaggle/temp/ATRA/model/atra-4b
export ATRA_SEQ=2048
# fp32: no autocast, no gradient scaler, nothing that can disagree about
# dtypes. Slower than fp16 AMP, and the 4-bit matmuls still run in float16
# inside bitsandbytes. Set ATRA_AMP=fp16 to try the faster path.
export ATRA_AMP=off
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
python train.py --config config/default.yaml --output /kaggle/working/runs/atra-4b 2>&1 | tee /kaggle/working/train.log
cat /kaggle/working/runs/atra-4b/manifest.json

## 6. Merge the adapter and convert to GGUF

The merge loads the full-precision base (~8 GB fp16) — fine on Kaggle's RAM/VRAM. Both checkouts live in `/kaggle/temp`, which is not part of the notebook's output, so only the adapter and the export are saved. `q8_0` needs only llama.cpp's Python converter (no build). For `q4_k_m` (smaller, ~2.5 GB, what a 6 GB laptop GPU wants) the notebook also builds `llama-quantize`; that takes a few minutes and is optional.

In [ ]:
%%bash
set -e
cd /kaggle/temp
rm -rf llama.cpp
git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
# Only the converter's own package. Its requirements file pins a CPU build of
# torch, which replaced Kaggle's CUDA torch on the 2026-09-20 run; everything
# else the converter imports (numpy, safetensors, transformers) is present.
pip install -q --no-deps gguf
# Optional: build the quantizer for q4_k_m (a few minutes).
cmake -S /kaggle/temp/llama.cpp -B /kaggle/temp/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
cmake --build /kaggle/temp/llama.cpp/build --target llama-quantize -j > /dev/null
ls /kaggle/temp/llama.cpp/build/bin | grep quantize

In [ ]:
%%bash
set -e
cd /kaggle/temp/ATRA/model/atra-4b
# Kaggle ships torchao 0.10.0. peft's LoRA dispatcher calls
# is_torchao_available() for every module it injects, and that helper raises
# on a version below 0.16.0 rather than returning False. Training does not
# reach it (the bitsandbytes dispatcher matches first on a 4-bit base); the
# merge loads the base in fp16, so it does. Neither path uses torchao.
pip uninstall -q -y torchao || true
python export.py --adapter /kaggle/working/runs/atra-4b --out /kaggle/working/atra-export   --llama-cpp /kaggle/temp/llama.cpp --gguf q4_k_m
ls -la /kaggle/working/atra-export
cat /kaggle/working/atra-export/export-manifest.json

## 7. Download

Everything you need is under `/kaggle/working/atra-export`:

- `atra-4b-q4_k_m.gguf` + `Modelfile` → on your machine: `ollama create atra-4b -f Modelfile`, then point the ATRA runtime at Ollama (`ATRA_LLM_KIND=ollama`, `ATRA_LLM_MODEL=atra-4b`).
- `merged/` → the fp16 safetensors, if you want to serve with vLLM or llama-server.
- `export-manifest.json` and `/kaggle/working/runs/atra-4b/manifest.json` → keep both; they are the provenance chain the model card requires.

Then, on your machine, run the evaluation suite against the served model before calling it anything:

```
cd model/atra-4b
python evaluate.py --model http://127.0.0.1:11434 --model-name atra-4b-v0 --out eval-atra-4b-v0.json
```

`evaluate.py` exits non-zero if any threshold in `config/default.yaml` is missed. Only a passing evaluation changes the model status from **UNTRAINED** to a versioned release, and only after the manifest, the evaluation output and the GGUF hash are committed together.

In [ ]:
%%bash
cd /kaggle/working
du -sh atra-export runs 2>/dev/null || true
# Optional: a single archive of the small artefacts (adapter + manifests), not the GGUF.
tar czf atra-4b-adapter.tgz -C /kaggle/working runs/atra-4b atra-export/export-manifest.json atra-export/Modelfile 2>/dev/null && ls -la atra-4b-adapter.tgz